# Semantic Search Engine
Build a search engine that ranks documents for a user query. This notebook demonstrates a TF-IDF lexical baseline and an optional embedding-based semantic search implementation.

**Features:** sample documents, ranked top-k results, optional Sentence-Transformers embeddings, a small Recall@k evaluation, and saving a local index.

> The example collection is synthetic and small. Evaluation metrics are for demonstrating the workflow, not proving production-quality search.

## 1. Install dependencies
Run in Google Colab or Jupyter. The embedding package is optional.

In [ ]:
%pip install -q pandas numpy scikit-learn
# Optional for embedding search: %pip install -q sentence-transformers

## 2. Imports

In [ ]:
from pathlib import Path
import json
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

OUTPUT_DIR = Path("semantic_search_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("Setup complete.")

## 3. Sample document collection

In [ ]:
documents = [
    {"id":"doc01","title":"Password Reset","category":"IT Support","text":"To reset your password, open account settings, choose security, and select reset password. Use a strong unique password."},
    {"id":"doc02","title":"Remote Work Policy","category":"HR","text":"Employees may work remotely up to three days per week with manager approval. Teams should remain available during core hours."},
    {"id":"doc03","title":"Expense Reimbursement","category":"Finance","text":"Submit business expense receipts through the finance portal within thirty days. Approved reimbursements are paid in the next cycle."},
    {"id":"doc04","title":"Vacation Leave","category":"HR","text":"Employees request annual leave through the HR system. Managers review requests based on team coverage and available leave balance."},
    {"id":"doc05","title":"Laptop Security","category":"IT Support","text":"Lock your laptop when away from your desk. Install approved security updates and report lost devices to IT immediately."},
    {"id":"doc06","title":"Travel Booking","category":"Finance","text":"Business travel must be approved before booking. Use the preferred travel provider and follow company travel expense rules."},
    {"id":"doc07","title":"Customer Refunds","category":"Customer Service","text":"Refund requests are reviewed against the return policy. Eligible refunds are processed to the original payment method."},
    {"id":"doc08","title":"Multi-Factor Authentication","category":"IT Support","text":"Enable multi-factor authentication to add a second verification step when signing in to your account."},
    {"id":"doc09","title":"Parental Leave","category":"HR","text":"Eligible employees can request parental leave through HR. Contact the people team to confirm eligibility and required documents."},
    {"id":"doc10","title":"Invoice Payment Terms","category":"Finance","text":"Supplier invoices should include a valid purchase order and payment details. Finance reviews complete invoices according to payment terms."},
]
docs_df = pd.DataFrame(documents)
display(docs_df)

## 4. Build a TF-IDF lexical-search baseline

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
document_texts = (docs_df["title"] + ". " + docs_df["text"]).tolist()
tfidf_matrix = tfidf.fit_transform(document_texts)
print("Indexed documents:", tfidf_matrix.shape[0])
print("Vocabulary size:", len(tfidf.vocabulary_))

## 5. Ranked search

In [ ]:
def lexical_search(query: str, top_k: int = 5) -> pd.DataFrame:
    query = str(query).strip()
    if not query:
        raise ValueError("Query must not be empty.")
    query_vector = tfidf.transform([query])
    scores = cosine_similarity(query_vector, tfidf_matrix).ravel()
    ranked = np.argsort(scores)[::-1][:max(1, top_k)]
    results = docs_df.iloc[ranked][["id", "title", "category", "text"]].copy()
    results["score"] = scores[ranked]
    return results.reset_index(drop=True)

lexical_search("How do I make my account safer when I sign in?", top_k=5)

## 6. Optional embedding-based semantic search
Install `sentence-transformers` and uncomment the code below. The model downloads the first time, requiring internet access. Embeddings can match related wording even when the query does not repeat the document's exact keywords.

In [ ]:
# Optional: install with %pip install -q sentence-transformers
# from sentence_transformers import SentenceTransformer
#
# model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# document_embeddings = model.encode(
#     document_texts, normalize_embeddings=True, show_progress_bar=True
# )
#
# def semantic_search(query: str, top_k: int = 5) -> pd.DataFrame:
#     if not str(query).strip():
#         raise ValueError("Query must not be empty.")
#     query_embedding = model.encode([query], normalize_embeddings=True)[0]
#     scores = document_embeddings @ query_embedding
#     ranked = np.argsort(scores)[::-1][:max(1, top_k)]
#     results = docs_df.iloc[ranked][["id", "title", "category", "text"]].copy()
#     results["score"] = scores[ranked]
#     return results.reset_index(drop=True)
#
# semantic_search("How can I add extra protection to login?", top_k=5)

## 7. Try multiple queries

In [ ]:
queries = [
    "How can I protect my login?",
    "What is the process for taking time off?",
    "How do I claim money spent on a business trip?",
    "What should I do if my work computer is stolen?",
]
for query in queries:
    print("\nQUERY:", query)
    display(lexical_search(query, top_k=3)[["id", "title", "category", "score"]])

## 8. Small retrieval evaluation
Recall@k checks whether the expected relevant document appears among the first k results. The hand-labelled examples below are only a demonstration.

In [ ]:
evaluation_queries = [
    {"query": "How can I add extra protection to login?", "relevant_ids": {"doc08"}},
    {"query": "How do I ask for time away from work?", "relevant_ids": {"doc04"}},
    {"query": "How do I get money back for work purchases?", "relevant_ids": {"doc03"}},
    {"query": "How should I arrange a work trip?", "relevant_ids": {"doc06"}},
]

def recall_at_k(search_fn, query, relevant_ids, k=3):
    results = search_fn(query, top_k=k)
    retrieved = set(results["id"].tolist())
    return len(retrieved & relevant_ids) / max(1, len(relevant_ids))

eval_rows = []
for item in evaluation_queries:
    score = recall_at_k(lexical_search, item["query"], item["relevant_ids"], k=3)
    eval_rows.append({
        "query": item["query"],
        "relevant_ids": ", ".join(sorted(item["relevant_ids"])),
        "recall_at_3": score,
    })
eval_df = pd.DataFrame(eval_rows)
display(eval_df)
print("Mean Recall@3:", round(eval_df["recall_at_3"].mean(), 3))

## 9. Save the local index and documents
Only load pickle files from trusted sources. This is a simple local demo index.

In [ ]:
index_path = OUTPUT_DIR / "tfidf_index.pkl"
documents_path = OUTPUT_DIR / "documents.json"
with index_path.open("wb") as f:
    pickle.dump({"vectorizer": tfidf, "matrix": tfidf_matrix}, f)
documents_path.write_text(json.dumps(documents, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved:", index_path.resolve())
print("Saved:", documents_path.resolve())

## 10. Next steps
- Compare lexical results with embedding-based semantic search.
- Add hybrid search combining lexical and dense-vector scores.
- Add metadata filters, deduplication, and incremental updates.
- Evaluate with a larger relevance-labelled dataset using Recall@k, MRR, and nDCG.
- Add a Streamlit UI or FastAPI endpoint.
- Consider a vector database, reranker, and access controls for production.